# 1.5 Personal Chef — A Complete Agent Application

## What this notebook demonstrates

This is the **capstone project** for Module 1. It combines every concept from the previous notebooks into one working application:

| Concept | Where it's used |
|---|---|
| **Agent** | `create_agent()` — the orchestration backbone |
| **System prompt** | Chef persona + task description |
| **Tool** | `web_search` — live recipe lookup via Tavily |
| **Memory** | `InMemorySaver` — remembers ingredients across turns |
| **Multi-turn conversation** | `thread_id` + `config` — user can follow up |

### The application

A **Personal Chef agent** that:
1. Takes a list of ingredients the user has at home
2. Searches the web for real recipes using those ingredients
3. Suggests options and provides full instructions on request
4. Remembers the conversation — no need to repeat the ingredient list

This is a pattern you can reuse for any domain: swap "chef" for "travel agent", "fitness coach", "financial advisor", etc.

---

### Architecture overview

```
User: "I have chicken and rice"
         │
         ▼
    ┌──────────────────────────────────────┐
    │            Agent Loop                │
    │  ┌─────────────────────────────┐     │
    │  │  System Prompt (Chef persona)│     │
    │  └─────────────────────────────┘     │
    │                 │                    │
    │          ┌──────▼──────┐             │
    │          │   LLM Brain  │             │
    │          └──────┬──────┘             │
    │                 │ decides to search  │
    │          ┌──────▼──────┐             │
    │          │  web_search  │ → Tavily   │
    │          │  (tool)      │ ← results  │
    │          └──────┬──────┘             │
    │                 │                    │
    │          ┌──────▼──────┐             │
    │          │  LLM Brain  │             │
    │          │ (summarises) │             │
    │          └──────┬──────┘             │
    └─────────────────┼────────────────────┘
                      │
         ┌────────────▼──────────────┐
         │  InMemorySaver (thread 1)  │  ← saves history
         └───────────────────────────┘
                      │
         ▼
    "Here are some recipes: 1) Chicken Fried Rice..."
```

In [ ]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================
# Requires in .env:
#   OPENAI_API_KEY=sk-...      (for the LLM)
#   TAVILY_API_KEY=tvly-...    (for web search)

from dotenv import load_dotenv

load_dotenv()

In [ ]:
# ============================================================
# CELL 2: Define the Web Search Tool
# ============================================================
# The same Tavily-backed tool from notebook 1.2.
# We define it here so this notebook is self-contained.
#
# The chef agent will use this tool to look up REAL recipes
# from the internet — not hallucinated ones from training data.
# This is critical for a recipe app where accuracy and safety
# (correct cooking times, temperatures) matter.
#
# Query examples the model will construct autonomously:
#   "chicken rice recipe easy"   → for ingredient-based search
#   "chicken fried rice recipe step by step"  → when user asks for details

from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()  # Reads TAVILY_API_KEY from env

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)

In [ ]:
# ============================================================
# CELL 3: Define the System Prompt
# ============================================================
# A good system prompt for a tool-enabled agent answers:
#   WHO:   "You are a personal chef"
#   WHAT:  "The user gives you ingredients"
#   HOW:   "Search the web for recipes" (explicitly instructs tool use)
#   OUTPUT:"Return suggestions and instructions on request"
#
# Note that we explicitly say "Using the web search tool" — this
# nudges the model to always search rather than rely on potentially
# stale or hallucinated recipes from training data.
#
# The phrase "if requested" for instructions is important:
# without it the model might dump the full recipe on every turn,
# making the conversation overwhelming. With it, it first suggests
# options and waits to be asked for details.

system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [ ]:
# ============================================================
# CELL 4: Build the Complete Agent
# ============================================================
# This one create_agent call wires together all the components:
#
#   model="gpt-5-nano"        → the LLM that reasons and generates text
#   tools=[web_search]        → the tool it can call (Tavily search)
#   system_prompt=system_prompt → the chef persona and task instructions
#   checkpointer=InMemorySaver() → per-thread conversation memory
#
# InMemorySaver is initialised fresh here — it starts empty.
# As we make invoke() calls with a thread_id, it fills up with
# the conversation history for that thread.

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()  # Enables multi-turn memory
)

In [ ]:
# ============================================================
# CELL 5: Turn 1 — Tell the Chef What Ingredients You Have
# ============================================================
# config binds this call to thread_id="1".
# We set config once and reuse it for all turns in this session.
#
# What the agent will do:
#   1. Read the system prompt (chef persona)
#   2. Read the user message (chicken + rice)
#   3. Decide to call web_search (the prompt says to search)
#   4. Formulate a query like "chicken rice leftover recipes"
#   5. Read Tavily results
#   6. Summarise 3-5 recipe options in natural language
#
# The InMemorySaver saves the full message history under thread_id="1"
# after this call — so the next call will remember the ingredients.

from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}  # Identifies this user's session

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config   # Attach to thread "1" — memory enabled
)

# Print the chef's recipe suggestions
print(response['messages'][-1].content)

In [ ]:
# ============================================================
# CELL 6: Inspect the Full Message Trace
# ============================================================
# Printing the full response dict shows everything:
#
#   response['messages'][0]  HumanMessage   → user's ingredient list
#   response['messages'][1]  AIMessage      → tool call (web_search)
#       .tool_calls shows the exact query the model chose
#   response['messages'][2]  ToolMessage    → raw Tavily JSON results
#   response['messages'][3]  AIMessage      → chef's recipe suggestions
#
# The Tavily results in messages[2] are real web content —
# you can inspect them to see exactly what the model read
# before composing the recipe suggestions.
#
# 💡 If the suggestions seem off, check messages[2]:
#    - Was the search query sensible?
#    - Did Tavily return relevant results?
#    - Did the model correctly extract the recipe names?

from pprint import pprint

pprint(response)

---
## Try It Yourself — Follow-Up Turns

Because memory is enabled, you can now continue the conversation naturally. Try adding a new cell with:

```python
response = agent.invoke(
    {"messages": [HumanMessage(content="Tell me how to make the fried rice")]},
    config   # Same thread_id — agent remembers the chicken and rice
)
print(response['messages'][-1].content)
```

The agent will:
1. Load the full history (ingredients + first suggestions)
2. Know you mean the chicken fried rice it suggested earlier
3. Search for step-by-step instructions
4. Return the full recipe

You never had to repeat "chicken and rice" — that's memory working.

---
## Summary — What Makes This a Real Application

| Component | Role in the app |
|---|---|
| `system_prompt` | Defines the chef persona and forces web search usage |
| `web_search` tool | Ensures recipes are real and current, not hallucinated |
| `InMemorySaver` | Lets the user say "make the first one" without repeating ingredients |
| `thread_id` | Isolates each user's conversation (in a real app: one per user session) |
| `[-1].content` | Always the final answer — safe regardless of how many tool calls happened |

### How to turn this into a production app

1. **Web API**: wrap the agent in a FastAPI endpoint — each request passes the `thread_id` from the user's session cookie
2. **Persistent memory**: swap `InMemorySaver` for `PostgresSaver` — memory survives server restarts
3. **Streaming**: use `agent.stream(..., stream_mode="messages")` so the user sees the recipe appear word-by-word
4. **Deploy**: see `langgraph.json` in this project — it configures the agent for deployment via LangGraph Cloud

### The `langgraph.json` file

The `langgraph.json` file in this project tells the LangGraph deployment platform:
- Which Python file and object to use as the agent (`1.5_personal_chef.py:agent`)
- Where to find dependencies (the current directory)
- Which `.env` file to use for API keys

This is how you go from a notebook prototype to a hosted API in one command: `langgraph up`.